# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
import os
from google import genai
from google.genai import types
from dotenv import load_dotenv 

load_dotenv()
client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

df = pd.read_csv('../data/ecommerce_faq_knowledge_base.csv') 

print("=== INFORMASI DATASET ===")
print(f"Jumlah entri (baris): {len(df)}")
print(f"Kolom yang tersedia: {df.columns.tolist()}")
print("\nDistribusi Kategori:")
print(df['category'].value_counts())
print("\nMissing values per kolom:")
print(df.isnull().sum())

df.head()

=== INFORMASI DATASET ===
Jumlah entri (baris): 59
Kolom yang tersedia: ['faq_id', 'category', 'question', 'answer']

Distribusi Kategori:
category
Pemesanan           9
Pengiriman          9
Pembayaran          8
Pengembalian        8
Akun                7
Produk              7
Promo               6
Customer Service    5
Name: count, dtype: int64

Missing values per kolom:
faq_id      0
category    0
question    0
answer      0
dtype: int64


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform...,"Pilih produk yang diinginkan, klik 'Tambah ke ..."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,Saat ini pemesanan hanya bisa dilakukan setela...
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah ...,Pesanan dapat dibatalkan melalui halaman 'Pesa...
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesa...,Penjual biasanya memproses pesanan dalam waktu...
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman se...,Perubahan alamat hanya bisa dilakukan sebelum ...


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
def get_embedding(text):
    # Menggunakan model embedding gemini
    result = client.models.embed_content(
        model='gemini-embedding-001',  
        contents=text
    )
    return result.embeddings[0].values

print("Sedang membuat embeddings... (proses ini mungkin memakan waktu beberapa detik)")
# membuat embedding untuk seluruh dokumen
embeddings = [get_embedding(doc) for doc in documents]

# konversi ke numpy array 
embeddings = np.array(embeddings).astype('float32')

# verifikasi 
print(f"Shape embeddings: {embeddings.shape}")

Sedang membuat embeddings... (proses ini mungkin memakan waktu beberapa detik)
Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

# verifikasi vektor
print(f"jumlah vektor yang berhasil disimpan di faiss: {index.ntotal}")
print(f"jumlah entri di knowledge base: {len(df)}")

assert index.ntotal == len(df), "jumlah vektor di faiss tidak sesuai dengan jumlah data"
print("jumlah vektor sesuai")

jumlah vektor yang berhasil disimpan di faiss: 59
jumlah entri di knowledge base: 59
jumlah vektor sesuai


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)
    
    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'faq_id': df.iloc[idx]['faq_id'],
            'category': df.iloc[idx]['category'],
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'distance': float(distances[0][rank])
        })
    return results

# uji coba
print("UJI COBA SEMANTIC SEARCH")
test_query_5 = "cara batalin pesanan gimana ya?"
results_5 = semantic_search(test_query_5, top_k=3)

print(f"Query: '{test_query_5}'\n")
for res in results_5:
    print(f"[Rank {res['rank']}] (ID: {res['faq_id']} | Kategori: {res['category']})")
    print(f"Pertanyaan FAQ : {res['question']}")
    print(f"Distance (L2)  : {res['distance']:.4f}")
    print("-" * 50)

UJI COBA SEMANTIC SEARCH
Query: 'cara batalin pesanan gimana ya?'

[Rank 1] (ID: 3 | Kategori: Pemesanan)
Pertanyaan FAQ : Bagaimana cara membatalkan pesanan yang sudah dibuat?
Distance (L2)  : 0.3051
--------------------------------------------------
[Rank 2] (ID: 24 | Kategori: Pengembalian)
Pertanyaan FAQ : Bagaimana prosedur pengembalian barang yang tidak sesuai?
Distance (L2)  : 0.4624
--------------------------------------------------
[Rank 3] (ID: 44 | Kategori: Produk)
Pertanyaan FAQ : Bagaimana jika produk yang saya pesan ternyata kehabisan stok?
Distance (L2)  : 0.4869
--------------------------------------------------


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:
def build_prompt(query, search_results):
    context_text = "\n\n".join([
        f"FAQ ID {r['faq_id']} (Kategori: {r['category']}):\nPertanyaan: {r['question']}\nJawaban: {r['answer']}" 
        for r in search_results
    ])
    
    prompt = f"""**PERSONA:**
Anda adalah asisten customer service e-commerce yang ramah, profesional, dan solutif.

**TASK:**
Tugas Anda adalah menjawab pertanyaan pengguna HANYA berdasarkan konteks FAQ yang disediakan di bawah ini. 

**CONTEXT:**
{context_text}

**FORMAT & ATURAN KETAT:**
1. Jawab dengan bahasa Indonesia yang natural dan sopan.
2. JANGAN menggunakan informasi dari luar konteks FAQ di atas.
3. Jika informasi yang ditanyakan pengguna TIDAK ADA di dalam konteks FAQ, Anda harus menjawab dengan jujur bahwa Anda belum memiliki informasi tersebut, lalu sarankan pengguna untuk menghubungi customer service manusia.

**USER QUERY:**
{query}

**JAWABAN ANDA:**
"""
    return prompt

# uji coba
test_query_6 = "apakah bisa bayar pakai cicilan?"
results_6 = semantic_search(test_query_6)
prompt_6 = build_prompt(test_query_6, results_6)

print("CONTOH PROMPT YANG DIKIRIM\n")
print(prompt_6)

CONTOH PROMPT YANG DIKIRIM

**PERSONA:**
Anda adalah asisten customer service e-commerce yang ramah, profesional, dan solutif.

**TASK:**
Tugas Anda adalah menjawab pertanyaan pengguna HANYA berdasarkan konteks FAQ yang disediakan di bawah ini. 

**CONTEXT:**
FAQ ID 20 (Kategori: Pembayaran):
Pertanyaan: Apakah saya bisa membayar dengan cicilan?
Jawaban: Fitur cicilan tersedia untuk transaksi dengan kartu kredit tertentu pada nominal minimal yang ditentukan pihak bank.

FAQ ID 22 (Kategori: Pembayaran):
Pertanyaan: Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
Jawaban: Saat ini sistem belum mendukung pembayaran gabungan (split payment) dalam satu transaksi.

FAQ ID 17 (Kategori: Pembayaran):
Pertanyaan: Metode pembayaran apa saja yang tersedia?
Jawaban: Kami mendukung transfer bank, kartu kredit/debit, e-wallet, dan pembayaran melalui minimarket rekanan.

**FORMAT & ATURAN KETAT:**
1. Jawab dengan bahasa Indonesia yang natural dan sopan.
2. JANGAN menggunakan infor

## 7. Mengintegrasikan LLM

In [7]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    
    prompt = build_prompt(query, search_results)
    
    response = client.models.generate_content(
        model='gemini-3.1-flash-lite',
        contents=prompt,
        config=types.GenerateContentConfig(
            temperature=0.3,
        )
    )
    
    return response.text, search_results

# uji coba
print("UJI COBA PIPELINE")
test_query_7 = "barang saya rusak waktu diterima, harus gimana?"
answer_7, sources_7 = generate_answer(test_query_7)

print(f"Query : {test_query_7}")
print(f"\nJawaban LLM:\n{answer_7}\n")
print("Sumber FAQ yang digunakan:")
for s in sources_7:
    print(f"- [ID {s['faq_id']}] {s['question']} (Distance: {s['distance']:.4f})")

UJI COBA PIPELINE


Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Query : barang saya rusak waktu diterima, harus gimana?

Jawaban LLM:
Halo! Mohon maaf atas ketidaknyamanan yang Anda alami terkait kondisi paket Anda.

Jika paket diterima dalam kondisi rusak, silakan segera ambil foto atau video kondisi paket tersebut saat diterima. Setelah itu, Anda dapat melaporkannya melalui menu komplain dalam waktu 2x24 jam sejak barang diterima. Semoga membantu!

Sumber FAQ yang digunakan:
- [ID 16] Apa yang harus dilakukan jika paket diterima dalam kondisi rusak? (Distance: 0.3043)
- [ID 54] Saya menerima barang yang salah, apa yang harus saya lakukan? (Distance: 0.4045)
- [ID 24] Bagaimana prosedur pengembalian barang yang tidak sesuai? (Distance: 0.4255)


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [9]:
import time
test_queries = [
    "Bagaimana cara membatalkan pesanan yang sudah dibuat?",
    "Berapa lama proses refund setelah pengembalian disetujui?",
    "Paket saya belum nyampe padahal udah lewat batas estimasi, gimana nih?",
    "Bisa nggak saya ganti baju yang kekecilan ini dengan ukuran yang lebih besar?",
    "Apakah platform ini menyediakan layanan pinjaman online (pinjol) tanpa jaminan?"
]

for i, q in enumerate(test_queries, 1):
    print(f"=== TEST CASE {i} ===")
    print(f"Query: {q}")
    answer, sources = generate_answer(q, top_k=2)
    
    print(f"\n[Jawaban AI]:\n{answer}")
    print("\n[Top FAQ Retrieved]:")
    for s in sources:
        print(f"- ID {s['faq_id']} | Distance: {s['distance']:.4f} | Q: {s['question']}")
    print("=" * 70 + "\n")
    time.sleep(3)

=== TEST CASE 1 ===
Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?

[Jawaban AI]:
Halo! Saya dengan senang hati membantu Anda.

Untuk membatalkan pesanan yang sudah dibuat, Anda dapat melakukannya melalui halaman 'Pesanan Saya' selama status pesanan Anda masih 'Diproses'. Namun, perlu diingat bahwa jika status pesanan sudah berubah menjadi 'Dikirim', maka pembatalan tidak dapat dilakukan lagi.

Apakah ada hal lain yang bisa saya bantu?

[Top FAQ Retrieved]:
- ID 3 | Distance: 0.2392 | Q: Bagaimana cara membatalkan pesanan yang sudah dibuat?
- ID 24 | Distance: 0.5069 | Q: Bagaimana prosedur pengembalian barang yang tidak sesuai?

=== TEST CASE 2 ===
Query: Berapa lama proses refund setelah pengembalian disetujui?

[Jawaban AI]:
Halo! Terima kasih telah menghubungi kami.

Mengenai pertanyaan Anda, dana refund biasanya akan diproses dalam waktu 3-7 hari kerja setelah barang pengembalian diterima dan diverifikasi oleh pihak penjual.

Apakah ada hal lain yang bisa saya bantu?

### Catatan Analisis Relevansi

Catatan Analisis Relevansi

Pengujian dilakukan dengan beberapa skenario query untuk menilai akurasi retrieval dan kualitas jawaban LLM.

Direct Query (Pertanyaan persis atau sangat mirip dengan teks FAQ).
Pada skenario ini, retrieval berjalan sangat akurat. Nilai distance yang dihasilkan sangat kecil karena teks query hampir sama dengan teks di knowledge base. LLM menghasilkan jawaban yang persis sesuai dengan konteks FAQ tanpa menambahkan informasi di luar konteks atau berhalusinasi.

Paraphrase Query (Pertanyaan dengan gaya bahasa sehari-hari atau sinonim).
Sistem semantic search berhasil menangkap makna dari pertanyaan pengguna meskipun menggunakan kata-kata yang berbeda dari FAQ asli. Nilai distance mungkin sedikit lebih besar dibanding direct query, namun FAQ yang terambil tetap relevan. LLM mampu merangkum jawaban dari konteks tersebut dengan gaya bahasa yang lebih natural dan empatik.

Out-of-Context Query (Pertanyaan yang tidak memiliki jawaban di knowledge base).
Sistem tetap mencoba mencari FAQ terdekat, namun nilai distance yang dihasilkan sangat besar karena topiknya tidak relevan. Berkat instruksi ketat pada prompt, LLM berhasil menolak menjawab dan mengaku tidak memiliki informasi tersebut, lalu menyarankan pengguna untuk menghubungi customer service manusia. Hal ini membuktikan bahwa sistem berhasil mencegah halusinasi.